In [89]:
!pip install transformers datasets torch --quiet
!pip install tqdm --quiet

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU, you are on CPU!")


CUDA available: True
GPU: Tesla T4


In [90]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from tqdm import tqdm
import random

# Your dev file with 200 samples
data_file = "/content/drive/MyDrive/ai project/dev_track_a.jsonl"


In [91]:
# Load samples
with open(data_file, "r", encoding="utf8") as f:
    samples = [json.loads(line) for line in f]

# Shuffle and split 90-10
random.shuffle(samples)
split_idx = int(0.9 * len(samples))
train_samples = samples[:split_idx]
val_samples = samples[split_idx:]

print(f"Train samples: {len(train_samples)}, Validation samples: {len(val_samples)}")


Train samples: 180, Validation samples: 20


In [92]:
class TrackADataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=256):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def encode_pair(self, anchor, choice):
        return self.tokenizer(
            anchor,
            choice,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc_a = self.encode_pair(s["anchor_text"], s["text_a"])
        enc_b = self.encode_pair(s["anchor_text"], s["text_b"])
        label = torch.tensor(int(s["text_a_is_closer"]), dtype=torch.long)
        return enc_a, enc_b, label


In [93]:
class CrossEncoderBERT(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        # Combine CLS_A, CLS_B, and their difference
        self.classifier = nn.Linear(hidden*3, 2)

    def forward(self, enc_a, enc_b):
        # Encode A
        out_a = self.encoder(
            input_ids=enc_a["input_ids"].squeeze(1),
            attention_mask=enc_a["attention_mask"].squeeze(1)
        )
        cls_a = out_a.last_hidden_state[:, 0]

        # Encode B
        out_b = self.encoder(
            input_ids=enc_b["input_ids"].squeeze(1),
            attention_mask=enc_b["attention_mask"].squeeze(1)
        )
        cls_b = out_b.last_hidden_state[:, 0]

        diff = cls_a - cls_b
        combined = torch.cat([cls_a, cls_b, diff], dim=1)

        return self.classifier(combined)


In [94]:
def train_model(train_samples, val_samples, model_name="bert-base-uncased",
                batch_size=4, epochs=5, lr=2e-5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = TrackADataset(train_samples, tokenizer)
    val_dataset = TrackADataset(val_samples, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    model = CrossEncoderBERT(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for enc_a, enc_b, label in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            enc_a = {k: v.to(device) for k, v in enc_a.items()}
            enc_b = {k: v.to(device) for k, v in enc_b.items()}
            label = label.to(device)

            logits = model(enc_a, enc_b)
            loss = criterion(logits, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} — Train Loss: {total_loss/len(train_loader):.4f}")
        evaluate(model, val_loader, device)

    return model, tokenizer


In [95]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for enc_a, enc_b, label in loader:
            enc_a = {k: v.to(device) for k, v in enc_a.items()}
            enc_b = {k: v.to(device) for k, v in enc_b.items()}
            label = label.to(device)

            logits = model(enc_a, enc_b)
            pred = torch.argmax(logits, dim=1)
            correct += int(pred == label)
            total += 1
    acc = correct / total
    print(f"Validation Accuracy: {correct}/{total} = {acc:.4f}")


In [96]:
model, tokenizer = train_model(
    train_samples,
    val_samples,
    model_name="bert-base-uncased",
    batch_size=4,
    epochs=10,
    lr=2e-5
)


Epoch 1: 100%|██████████| 45/45 [00:19<00:00,  2.33it/s]


Epoch 1 — Train Loss: 0.7374
Validation Accuracy: 8/20 = 0.4000


Epoch 2: 100%|██████████| 45/45 [00:19<00:00,  2.30it/s]


Epoch 2 — Train Loss: 0.6676
Validation Accuracy: 11/20 = 0.5500


Epoch 3: 100%|██████████| 45/45 [00:18<00:00,  2.45it/s]


...

Epoch 8 — Train Loss: 0.0040
Validation Accuracy: 13/20 = 0.6500


Epoch 9: 100%|██████████| 45/45 [00:18<00:00,  2.47it/s]


Epoch 9 — Train Loss: 0.0035
Validation Accuracy: 13/20 = 0.6500


Epoch 10: 100%|██████████| 45/45 [00:18<00:00,  2.49it/s]


Epoch 10 — Train Loss: 0.0025
Validation Accuracy: 12/20 = 0.6000
